Ingesta de datos del DENUE

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# 1. Crear o recuperar la SparkSession local
spark = SparkSession.builder \
    .appName("AnalizadorOportunidadesEdomex") \
    .master("local[*]") \
    .getOrCreate()

print("==================================================")
print("¡SparkSession inicializada correctamente en local!")
print("==================================================\n")

# 2. Esquema explícito con TODAS las columnas del DENUE INEGI
esquema_denue = StructType([
    StructField("id", StringType(), True),
    StructField("clelee", StringType(), True),
    StructField("nom_estab", StringType(), True),
    StructField("raz_social", StringType(), True),
    StructField("codigo_act", StringType(), True),
    StructField("nombre_act", StringType(), True),
    StructField("per_ocu", StringType(), True),
    StructField("tipo_vial", StringType(), True),
    StructField("nom_vial", StringType(), True),
    StructField("numero_ext", StringType(), True),
    StructField("letra_ext", StringType(), True),
    StructField("edificio", StringType(), True),
    StructField("edificio_e", StringType(), True),
    StructField("numero_int", StringType(), True),
    StructField("letra_int", StringType(), True),
    StructField("tipo_asent", StringType(), True),
    StructField("nomb_asent", StringType(), True),
    StructField("tipoCenCom", StringType(), True),
    StructField("nom_CenCom", StringType(), True),
    StructField("num_local", StringType(), True),
    StructField("cod_postal", StringType(), True),
    StructField("cve_ent", StringType(), True),
    StructField("entidad", StringType(), True),
    StructField("cve_mun", StringType(), True),
    StructField("municipio", StringType(), True),
    StructField("cve_loc", StringType(), True),
    StructField("localidad", StringType(), True),
    StructField("ageb", StringType(), True),
    StructField("manzana", StringType(), True),
    StructField("telefono", StringType(), True),
    StructField("correoelec", StringType(), True),
    StructField("www", StringType(), True),
    StructField("tipoUniEco", StringType(), True),
    StructField("latitud", DoubleType(), True),
    StructField("longitud", DoubleType(), True),
    StructField("fecha_alta", StringType(), True)
])

print("Esquema explícito definido correctamente.")

¡SparkSession inicializada correctamente en local!

Esquema explícito definido correctamente.


Cuando trabajas con un dataset nuevo, nunca debes adivinar las columnas ni escribirlas a ciegas. Hay una técnica estándar para explorar el archivo primero usando Python nativo o Pandas (que es muy rápido para examinar solo las primeras filas) y luego construir tu esquema de PySpark con esa información.

Aquí te muestro los 3 métodos paso a paso para que puedas aplicarlo a cualquier CSV en el futuro.

In [ ]:
Método 1: Leer solo los encabezados con Python puro (El más rápido y ligero)
No necesitas cargar millones de filas en memoria solo para ver los nombres de las columnas. Python tiene la librería nativa csv 
para leer únicamente la primera línea del archivo.


In [ ]:
import csv

# Reemplaza por la ruta de tu nuevo archivo CSV
ruta_archivo = "Datos/conjunto_de_datos/denue_inegi_15_.csv" # O la primera parte de tu CSV

with open(ruta_archivo, mode="r", encoding="latin1") as archivo:
    lector_csv = csv.reader(archivo)
    # leemos únicamente la primera fila (los encabezados)
    encabezados = next(lector_csv) 

print(f"Total de columnas encontradas: {len(encabezados)}\n")
print("Lista de columnas:")
for i, columna in enumerate(encabezados, start=1):
    print(f"{i}. {columna}")

¿Qué lograste con esto?
Obtuviste la lista exacta de nombres de columnas en menos de 1 segundo sin saturar la memoria. Con esa lista impresa, ya sabes exactamente qué nombres poner en tu StructField("nombre_columna", StringType(), True).

Método 2: Dejar que PySpark lea 5 filas con inferSchema para que te genere el código borrador
Si no quieres escribir StructField(...) a mano para 35 o 50 columnas, puedes pedirle a PySpark que lea únicamente una muestra muy pequeña del archivo, que adivine el esquema y te imprima el resultado.

In [ ]:
# 1. Leemos solo una muestra pequeña del CSV activando inferSchema
df_muestra = spark.read \
    .option("header", "true") \
    .option("encoding", "latin1") \
    .option("inferSchema", "true") \
    .csv("Datos/conjunto_de_datos/*.csv")

# 2. Le pedimos a PySpark que nos imprima el esquema en formato de código
print(df_muestra.schema)

¿Qué te devuelve df_muestra.schema?
Te imprime en texto la estructura StructType([StructField(...), ...]) ya armada. Tú solo tienes que:

Copiar ese texto devuelto.

Cambiar los tipos de datos que Spark haya puesto como IntegerType o DoubleType a StringType en aquellas columnas que sean claves, códigos postales o IDs (para no perder ceros a la izquierda).

Método 3: Inspeccionar las primeras filas con Pandas
Pandas es ideal para "ojear" los datos rápidamente. Puedes leer solo las primeras 3 filas usando el parámetro nrows=3:

In [ ]:
import pandas as pd

# Leer solo 3 filas del CSV
df_pandas = pd.read_csv("Datos/conjunto_de_datos/denue_inegi_15_.csv", encoding="latin1", nrows=3)

# Ver los nombres de las columnas y qué tipo de datos detectó Pandas
print(df_pandas.dtypes)

Esto te muestra en pantalla una lista clara de cada columna acompañada del tipo de dato que contiene en esas primeras filas (object equivale a texto/string, float64 a decimales, int64 a enteros).

En resumen: Tu flujo de trabajo para un CSV nuevo siempre será:
Inspeccionar: Usas el Método 1 (Python nativo) o Método 3 (Pandas) para conocer los nombres de las columnas y ver qué tipo de información traen (si hay claves, coordenadas, fechas, etc.).

Diseñar: Decides cuáles columnas necesitan ser estrictamente texto (StringType) para no perder formato (como folios, códigos postales, claves de municipio) y cuáles sí pueden ser números (DoubleType).

Construir: Armas tu StructType en PySpark sabiendo exactamente qué esperar.

**Leer archivos CSV con PySpark**

Spark leerá en paralelo todos los archivos .csv contenidos dentro de la carpeta Datos/conjunto_de_datos gracias al comodín *.csv

In [2]:
import os

# Verificar qué archivos hay en la carpeta
archivos = os.listdir("Datos/conjunto_de_datos")
print("Archivos encontrados en la carpeta:")
for f in archivos:
    print("-", f)

Archivos encontrados en la carpeta:
- denue_10101313549_1_15.csv
- denue_part_1.csv
- denue_part_10.csv
- denue_part_11.csv
- denue_part_12.csv
- denue_part_13.csv
- denue_part_14.csv
- denue_part_15.csv
- denue_part_16.csv
- denue_part_17.csv
- denue_part_18.csv
- denue_part_19.csv
- denue_part_20.csv
- denue_part_21.csv
- denue_part_22.csv
- denue_part_23.csv
- denue_part_24.csv
- denue_part_25.csv
- denue_part_26.csv
- denue_part_27.csv
- denue_part_28.csv
- denue_part_29.csv
- denue_part_3.csv
- denue_part_30.csv
- denue_part_31.csv
- denue_part_32.csv
- denue_part_33.csv
- denue_part_34.csv
- denue_part_35.csv
- denue_part_36.csv
- denue_part_37.csv
- denue_part_38.csv
- denue_part_39.csv
- denue_part_4.csv
- denue_part_40.csv
- denue_part_5.csv
- denue_part_6.csv
- denue_part_7.csv
- denue_part_8.csv
- denue_part_9.csv


In [3]:
# Reemplaza 'nombre_de_tu_archivo.csv' por uno real que haya salido en la lista anterior
ruta_un_archivo = "Datos/conjunto_de_datos/denue_part_1.csv" 

df_denue_raw = spark.read \
    .option("header", "true") \
    .option("encoding", "latin1") \
    .schema(esquema_denue) \
    .csv(ruta_un_archivo)

print("¡Lectura configurada al instante!")

¡Lectura configurada al instante!


In [6]:
import glob

# 1. Obtener la lista exacta de todos los archivos .csv en la carpeta
lista_archivos_csv = glob.glob("Datos/conjunto_de_datos/*.csv")

print(f"Se encontraron {len(lista_archivos_csv)} archivos CSV para procesar:")
for archivo in lista_archivos_csv[:5]: # Imprime los primeros 5 para verificar
    print(" -", archivo)

# 2. Leer la lista completa de archivos en PySpark
df_denue_raw = spark.read \
    .option("header", "true") \
    .option("encoding", "ISO-8859-1") \
    .schema(esquema_denue) \
    .csv(lista_archivos_csv)

print("\n¡Lectura de TODOS los archivos configurada exitosamente!")

Se encontraron 40 archivos CSV para procesar:
 - Datos/conjunto_de_datos\denue_10101313549_1_15.csv
 - Datos/conjunto_de_datos\denue_part_1.csv
 - Datos/conjunto_de_datos\denue_part_10.csv
 - Datos/conjunto_de_datos\denue_part_11.csv
 - Datos/conjunto_de_datos\denue_part_12.csv

¡Lectura de TODOS los archivos configurada exitosamente!


En PySpark esto ocurre gracias al concepto de Evaluación Perezosa (Lazy Evaluation):

Spark no ha cargado los datos a la memoria RAM todavía. En este paso, Spark solo leyó la estructura de los archivos y creó un mapa de ruta interno llamado DAG (Grafo Dirigido Acíclico).

Es como si le hubieras dado la lista de supermercado a un asistente: en este punto solo anotó qué va a comprar y dónde está cada cosa, pero todavía no ha ido a los pasillos a recoger los productos.

Paso 4: Validar la ingesta mediante una "Acción"
Para poner a trabajar el motor distribuido de Spark y verificar que todo se leyó correctamente, aplicaremos nuestras primeras Acciones. A diferencia de las lecturas o filtros (que son Transformaciones y se acumulan en el DAG), las Acciones obligan a Spark a procesar los datos de verdad y devolver un resultado.

In [7]:
# 1. Contar el número total de registros a nivel nacional (Acción: .count())
# Nota: Como aquí sí procesará todos los archivos, esta celda tomará unos segundos en calcular.
total_registros = df_denue_raw.count()
print(f"Total de establecimientos registrados en todo el país: {total_registros:,}\n")

# 2. Inspeccionar las primeras 5 filas con columnas clave (Acción: .show())
df_denue_raw.select(
    "id", 
    "nom_estab", 
    "codigo_act", 
    "nombre_act", 
    "entidad", 
    "municipio"
).show(5, truncate=False)

Total de establecimientos registrados en todo el país: 16,337,354

+-------+-------------------+----------+----------+-------+----------------+
|id     |nom_estab          |codigo_act|nombre_act|entidad|municipio       |
+-------+-------------------+----------+----------+-------+----------------+
|4247672|VALLE DE BRAVO     |0174      |800       |NULL   |6 a 10 personas |
|4247666|AMANALCO           |0046      |026       |NULL   |0 a 5 personas  |
|4247723|HUIXQUILUCAN       |0168      |800       |NULL   |31 a 50 personas|
|1512491|NAUCALPAN DE JUÁREZ|1169      |030       |NULL   |6 a 10 personas |
|4247674|VALLE DE BRAVO     |016A      |000       |NULL   |6 a 10 personas |
+-------+-------------------+----------+----------+-------+----------------+
only showing top 5 rows


El DENUE del INEGI tiene dos tipos de archivos CSV distintos en su portal:

Archivos CSV generales por estado (que traen ~36 columnas tradicionales).

Archivos CSV específicos o de versiones distintas que pueden traer las columnas en un orden ligeramente diferente o con campos extra.

Cuando le dijimos a Spark que leeyera todos los archivos de la carpeta (Datos/conjunto_de_datos/*.csv), combinó archivos que tienen diferentes estructuras, provocando que los datos se descalificaran contra el esquema_denue que definimos.

¿Cómo solucionamos esto y dejamos los datos limpios?
Vamos a hacer una inspección de qué archivos hay dentro de Datos/conjunto_de_datos/ para entender qué descargamos exactamente.

In [8]:
import csv
import glob

# Inspeccionar las columnas reales de cada archivo CSV en la carpeta
archivos = glob.glob("Datos/conjunto_de_datos/*.csv")

print(f"Total de archivos CSV encontrados: {len(archivos)}\n")

for ruta in archivos:
    with open(ruta, mode="r", encoding="ISO-8859-1") as f:
        lector = csv.reader(f)
        primer_fila = next(lector)
        nombre_archivo = ruta.split("\\")[-1].split("/")[-1]
        print(f"Archivo: {nombre_archivo} -> Contiene {len(primer_fila)} columnas")

Total de archivos CSV encontrados: 40

Archivo: denue_10101313549_1_15.csv -> Contiene 28 columnas
Archivo: denue_part_1.csv -> Contiene 24 columnas
Archivo: denue_part_10.csv -> Contiene 41 columnas
Archivo: denue_part_11.csv -> Contiene 41 columnas
Archivo: denue_part_12.csv -> Contiene 41 columnas
Archivo: denue_part_13.csv -> Contiene 41 columnas
Archivo: denue_part_14.csv -> Contiene 41 columnas
Archivo: denue_part_15.csv -> Contiene 41 columnas
Archivo: denue_part_16.csv -> Contiene 41 columnas
Archivo: denue_part_17.csv -> Contiene 41 columnas
Archivo: denue_part_18.csv -> Contiene 41 columnas
Archivo: denue_part_19.csv -> Contiene 41 columnas
Archivo: denue_part_20.csv -> Contiene 42 columnas
Archivo: denue_part_21.csv -> Contiene 42 columnas
Archivo: denue_part_22.csv -> Contiene 42 columnas
Archivo: denue_part_23.csv -> Contiene 42 columnas
Archivo: denue_part_24.csv -> Contiene 42 columnas
Archivo: denue_part_25.csv -> Contiene 42 columnas
Archivo: denue_part_26.csv -> Conti

¿Cómo soluciona un Data Engineer esto en la vida real?
Para solucionar esto de manera profesional en PySpark, no le pasamos el esquema rígido al read, sino que dejamos que Spark lea los nombres de los encabezados de cada archivo y luego seleccionamos/unimos las columnas por su nombre verdadero, no por su posición.

PySpark tiene una opción perfecta para esto: .option("mergeSchema", "true") o, mejor aún, dejar que lea los encabezados e inferir solo los nombres de columnas.

El nuevo plan de ingesta limpio y robusto
Copia y reemplaza el código del Paso 3 con el siguiente bloque.

En este enfoque:

Le decimos a Spark que use la primera fila de cada CSV como el nombre real de la columna (header=True).

No le imponemos una posición fija por índice, evitando el desfasamiento.

Luego, estandarizamos y seleccionamos únicamente las columnas clave que realmente necesitamos para el proyecto.

In [9]:
import glob

# 1. Obtener la lista de archivos CSV
lista_archivos_csv = glob.glob("Datos/conjunto_de_datos/*.csv")

# 2. Leer los CSV dejando que Spark asocie los datos por NOMBRE DE COLUMNA (header)
# Nota: Al no imponer el StructType rígido, no habrá desfasamiento por posición.
df_denue_raw = spark.read \
    .option("header", "true") \
    .option("encoding", "ISO-8859-1") \
    .csv(lista_archivos_csv)

print("¡Lectura por encabezados completada exitosamente!")

¡Lectura por encabezados completada exitosamente!


Verificación inmediata
Ahora ejecuta la validación en una celda para asegurarnos de que el desfasamiento desapareció y cada campo cayó en su lugar:

In [10]:
# Probar seleccionar las columnas por su nombre real
df_denue_raw.select(
    "id", 
    "nom_estab", 
    "codigo_act", 
    "nombre_act", 
    "entidad", 
    "municipio"
).show(5, truncate=False)

{"ts": "2026-09-27 18:41:00.099", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `id` cannot be resolved. Did you mean one of the following? [`Latitud`, `Longitud`, `Localidad`, `Manzana`, `Municipio`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.DirectMethodHandleAccessor.invoke(DirectMethodHandleAccessor.java", "line": "103)", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o78.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `id` cannot be resolved. Did you mean one of the following? [`Latitud`, `Longitud`, `Localidad`, `Manzana`, `Municipio`]. SQLSTATE: 42703;\n'Project ['id, 'nom_estab, 'codigo_act, 'nombre_act, 'entidad, municipio#239]\n+- Relation [Llave DEN

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `id` cannot be resolved. Did you mean one of the following? [`Latitud`, `Longitud`, `Localidad`, `Manzana`, `Municipio`]. SQLSTATE: 42703;
'Project ['id, 'nom_estab, 'codigo_act, 'nombre_act, 'entidad, municipio#239]
+- Relation [Llave DENUE#237,Entidad federativa#238,Municipio#239,Localidad#240,Área geoestadística básica #241,Manzana#242,Nombre de la Unidad Económica#243,Razón social#244,Tipo de vialidad#245,Nombre de la vialidad#246,Número exterior o kilómetro#247,Edificio, piso o nivel#248,Número o letra interior#249,Tipo de asentamiento humano#250,Nombre de asentamiento humano#251,Código Postal#252,Número de teléfono 1#253,Código de la clase de actividad SCIAN#254,Nombre de clase de la actividad#255,Corredor industrial, centro comercial o mercado público#256,Número de local#257,Correo electrónico 1#258,Sitio en Internet#259,Tipo de establecimiento#260,Descripcion estrato perso

La Solución: Definir el Esquema con los Nombres Reales Largos
En lugar de pelear con las abreviaturas (id, nom_estab), le damos a PySpark el esquema explícito usando los nombres exactos que vienen impresos en el encabezado de tus archivos CSV.

Sigue estos 3 pasos para dejar la ingesta perfecta:

Paso 1: Actualizar la celda del Esquema Explícito
Copia este bloque en la celda donde definiste el esquema (esquema_denue):

In [11]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# Esquema basado en los nombres de encabezado reales que traen tus archivos
esquema_denue_real = StructType([
    StructField("Llave DENUE", StringType(), True),
    StructField("Entidad federativa", StringType(), True),
    StructField("Municipio", StringType(), True),
    StructField("Localidad", StringType(), True),
    StructField("Área geoestadística básica", StringType(), True),
    StructField("Manzana", StringType(), True),
    StructField("Nombre de la Unidad Económica", StringType(), True),
    StructField("Razón social", StringType(), True),
    StructField("Tipo de vialidad", StringType(), True),
    StructField("Nombre de la vialidad", StringType(), True),
    StructField("Número exterior o kilómetro", StringType(), True),
    StructField("Edificio, piso o nivel", StringType(), True),
    StructField("Número o letra interior", StringType(), True),
    StructField("Tipo de asentamiento humano", StringType(), True),
    StructField("Nombre de asentamiento humano", StringType(), True),
    StructField("Código Postal", StringType(), True),
    StructField("Número de teléfono 1", StringType(), True),
    StructField("Código de la clase de actividad SCIAN", StringType(), True),
    StructField("Nombre de clase de la actividad", StringType(), True),
    StructField("Corredor industrial, centro comercial o mercado público", StringType(), True),
    StructField("Número de local", StringType(), True),
    StructField("Correo electrónico 1", StringType(), True),
    StructField("Sitio en Internet", StringType(), True),
    StructField("Tipo de establecimiento", StringType(), True),
    StructField("Descripcion estrato personal ocupado", StringType(), True),
    StructField("Latitud", DoubleType(), True),
    StructField("Longitud", DoubleType(), True),
    StructField("Fecha de incorporación al DENUE", StringType(), True)
])

print("¡Esquema con nombres reales configurado!")

¡Esquema con nombres reales configurado!


Paso 2: Leer los archivos usando este esquema real
En la celda del Paso 3, ejecuta la lectura con glob apuntando a las partes del DENUE:

In [12]:
import glob

# Seleccionar las partes principales del DENUE
lista_archivos = glob.glob("Datos/conjunto_de_datos/denue_part_*.csv")

df_denue_raw = spark.read \
    .option("header", "true") \
    .option("encoding", "ISO-8859-1") \
    .schema(esquema_denue_real) \
    .csv(lista_archivos)

print(f"Se leyeron {len(lista_archivos)} archivos correctamente.")

Se leyeron 39 archivos correctamente.


Paso 3: Renombrar o seleccionar con los nombres reales
Ahora ejecuta la inspección usando los nombres reales exactos de las columnas:

Python

In [13]:
df_denue_raw.select(
    "Llave DENUE", 
    "Nombre de la Unidad Económica", 
    "Código de la clase de actividad SCIAN", 
    "Nombre de clase de la actividad", 
    "Entidad federativa", 
    "Municipio"
).show(5, truncate=False)

+-----------------------------------+-----------------------------+-------------------------------------+-------------------------------+-----------------------------------+---------+
|Llave DENUE                        |Nombre de la Unidad Económica|Código de la clase de actividad SCIAN|Nombre de clase de la actividad|Entidad federativa                 |Municipio|
+-----------------------------------+-----------------------------+-------------------------------------+-------------------------------+-----------------------------------+---------+
|H AYUNTAMIENTO MUNICIPAL DE ACAMBAY|1                            |009                                  |0                              |H AYUNTAMIENTO MUNICIPAL DE ACAMBAY|222111   |
|MOLINO DE CHILES EL PIQUIN         |2                            |003                                  |0                              |NULL                               |311214   |
|PALETERIA GABI                     |7                            |016          

La Solución Definitiva (Garantizada)
En PySpark, la mejor manera de asegurarnos de que cada dato caiga exactamente en su columna correspondiente cuando el archivo traiga encabezados es:

Dejar que Spark lea los encabezados reales (sin imponer un StructType que desordene las posiciones).

Convertir las columnas que necesitemos al tipo de dato adecuado.

Copia y ejecuta estas celdas para resolver el desorden de una vez por todas:

Celda 1: Ingesta dinámica por nombre de encabezado

In [14]:
import glob

# 1. Obtenemos los archivos denue_part_*.csv
lista_archivos = glob.glob("Datos/conjunto_de_datos/denue_part_*.csv")

# 2. Leemos SIN forzar esquema para que interprete el orden REAL del CSV
df_denue_raw = spark.read \
    .option("header", "true") \
    .option("encoding", "ISO-8859-1") \
    .csv(lista_archivos)

print(f"Se leyeron {len(lista_archivos)} archivos. Columnas detectadas:")
print(df_denue_raw.columns)

Se leyeron 39 archivos. Columnas detectadas:
['Nombre de la unidad económica', 'Razón social', 'Código de la clase de actividad', 'Nombre de la clase de actividad', 'Personal ocupado (estrato)', 'Calle, avenida, andador, carretera, manzana u otro', 'Número exterior o km', 'Edificio, piso o nivel', 'Número o letra interior', 'Colonia, fraccionamiento, unidad habitacional o barrio', 'Corredor industrial, centro comercial o mercado público', 'Número de local', 'Código postal', 'Entidad federativa', 'Municipio', 'Localidad', 'Área geoestadística básica', 'Manzana', 'Número de teléfono', 'Correo electrónico', 'Sitio en Internet', 'Tipo de unidad económica', 'Latitud', 'Longitud']


Celda 2: Inspección limpia de los datos reales
Ahora haz la prueba de seleccionar las columnas exactas usando los nombres reales detectados por Spark:

In [16]:
# Inspección de datos con sus encabezados reales
df_denue_raw.select(
    "Llave DENUE", 
    "Nombre de la Unidad Económica", 
    "Código de la clase de actividad SCIAN", 
    "Nombre de clase de la actividad", 
    "Entidad federativa", 
    "Municipio"
).show(5, truncate=False)

{"ts": "2026-09-27 18:47:25.128", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Llave DENUE` cannot be resolved. Did you mean one of the following? [`Latitud`, `Localidad`, `Longitud`, `Manzana`, `Municipio`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.DirectMethodHandleAccessor.invoke(DirectMethodHandleAccessor.java", "line": "103)", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o121.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Llave DENUE` cannot be resolved. Did you mean one of the following? [`Latitud`, `Localidad`, `Longitud`, `Manzana`, `Municipio`]. SQLSTATE: 42703;\n'Project ['Llave DENUE, Nombre de la Unidad Económica#343, 'Código de la clase de 

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Llave DENUE` cannot be resolved. Did you mean one of the following? [`Latitud`, `Localidad`, `Longitud`, `Manzana`, `Municipio`]. SQLSTATE: 42703;
'Project ['Llave DENUE, Nombre de la Unidad Económica#343, 'Código de la clase de actividad SCIAN, 'Nombre de clase de la actividad, Entidad federativa#356, Municipio#357]
+- Relation [Nombre de la unidad económica#343,Razón social#344,Código de la clase de actividad#345,Nombre de la clase de actividad#346,Personal ocupado (estrato)#347,Calle, avenida, andador, carretera, manzana u otro#348,Número exterior o km#349,Edificio, piso o nivel#350,Número o letra interior#351,Colonia, fraccionamiento, unidad habitacional o barrio#352,Corredor industrial, centro comercial o mercado público#353,Número de local#354,Código postal#355,Entidad federativa#356,Municipio#357,Localidad#358,Área geoestadística básica#359,Manzana#360,Número de teléfono#3

El Diagnóstico Real del Problema
Tienes 40 archivos CSV.

Los archivos no son homogéneos en esquema:

Unos tienen 24 columnas, otros 28, otros 41 y otros 42.

Un grupo usa encabezados cortos (id, nom_estab, cve_ent, cve_mun).

Otro grupo usa encabezados largos en español (Llave DENUE, Nombre de la Unidad Económica, Entidad federativa, Municipio).

Por qué falla Spark si los lee juntos con un esquema fijo (StructType): Spark asigna los datos por la posición ordinal de la columna (índice 0, 1, 2...), no por el nombre del encabezado. Si el archivo A tiene la clave del municipio en la columna 3 y el archivo B la tiene en la columna 8, forzar una lectura conjunta provoca que los municipios terminen en el estrato de personal y los nombres de los negocios en el ID.

Por qué falla Spark si los lee juntos con .option("header", "true") sin esquema: Si le pasas una lista de archivos cuyas cabeceras no coinciden exactamente columna por columna, PySpark arroja AnalysisException: UNRESOLVED_COLUMN porque las firmas del catálogo de datos de cada archivo no son compatibles.

La Solución de Ingeniería de Datos (Paso a Paso)
En PySpark, cuando te enfrentas a un "Data Lake" desestructurado o con esquemas heterogéneos, la estrategia profesional consiste en:

Inspeccionar mediante Python las cabeceras de los 40 archivos.

Crear un diccionario de mapeo/alias (Homologación de Nombres): Traducir tanto los nombres cortos como los largos a un esquema estándar e unificado.

Leer archivo por archivo (o por grupos de esquemas idénticos), estandarizar sus columnas y unirlos con unionByName(allowMissingColumns=True).

unionByName de PySpark con allowMissingColumns=True alinea las columnas por su nombre estricto (sin importar en qué posición física venían en el CSV) y llena con NULL las columnas que falten en los archivos más pequeños.

Paso 1: Mapeador y Lector Homologado
Ejecuta el siguiente bloque de código en tu notebook. Este código implementa la lógica de homologación:

In [17]:
import glob
from pyspark.sql.functions import col

# 1. Diccionario de homologación (Mapea nombres largos y cortos al estándar del proyecto)
MAPEO_COLUMNAS = {
    # Llave/ID
    "id": "id",
    "Llave DENUE": "id",
    
    # Nombre del establecimiento
    "nom_estab": "nom_estab",
    "Nombre de la Unidad Económica": "nom_estab",
    
    # Razón Social
    "raz_social": "raz_social",
    "Razón social": "raz_social",
    
    # Código y nombre de actividad SCIAN
    "codigo_act": "codigo_act",
    "Código de la clase de actividad SCIAN": "codigo_act",
    "nombre_act": "nombre_act",
    "Nombre de clase de la actividad": "nombre_act",
    
    # Estrato de personal
    "per_ocu": "per_ocu",
    "Descripcion estrato personal ocupado": "per_ocu",
    
    # Ubicación geográfica
    "cod_postal": "cod_postal",
    "Código Postal": "cod_postal",
    "cve_ent": "cve_ent",
    "Entidad federativa": "entidad",
    "entidad": "entidad",
    "cve_mun": "cve_mun",
    "Municipio": "municipio",
    "municipio": "municipio",
    
    # Coordenadas
    "latitud": "latitud",
    "Latitud": "latitud",
    "longitud": "longitud",
    "Longitud": "longitud"
}

# 2. Obtener los 40 archivos CSV
archivos_csv = glob.glob("Datos/conjunto_de_datos/*.csv")
print(f"Iniciando procesamiento y alineación de {len(archivos_csv)} archivos...")

dataframes_estandarizados = []

# 3. Procesar cada archivo leyendo su propia cabecera y estandarizando sus columnas
for ruta in archivos_csv:
    # Leemos el archivo CSV individual usando su propia cabecera
    df_temp = spark.read \
        .option("header", "true") \
        .option("encoding", "ISO-8859-1") \
        .csv(ruta)
    
    # Renombrar únicamente las columnas que coincidan con nuestro mapa de estandarización
    for col_original in df_temp.columns:
        if col_original in MAPEO_COLUMNAS:
            df_temp = df_temp.withColumnRenamed(col_original, MAPEO_COLUMNAS[col_original])
    
    # Seleccionar solo las columnas estandarizadas que realmente nos interesan
    cols_presentes = [c for c in set(MAPEO_COLUMNAS.values()) if c in df_temp.columns]
    df_temp_filtrado = df_temp.select(cols_presentes)
    
    dataframes_estandarizados.append(df_temp_filtrado)

# 4. Unir todos los DataFrames alineándolos estrictamente POR NOMBRE DE COLUMNA
df_denue_raw = dataframes_estandarizados[0]
for df_siguiente in dataframes_estandarizados[1:]:
    df_denue_raw = df_denue_raw.unionByName(df_siguiente, allowMissingColumns=True)

print("¡Procesamiento masivo y homologación completados exitosamente!")

Iniciando procesamiento y alineación de 40 archivos...
¡Procesamiento masivo y homologación completados exitosamente!


Paso 2: Validación Inmediata del Resultado
Una vez que termine de ejecutarse el script anterior, ejecuta esta celda de validación:

In [18]:
# Contar el total consolidado de registros a nivel nacional
total_registros = df_denue_raw.count()
print(f"Total de registros consolidados a nivel nacional: {total_registros:,}\n")

# Mostrar las primeras 5 filas para verificar que CADA DATO está en su lugar
df_denue_raw.select(
    "id", 
    "nom_estab", 
    "codigo_act", 
    "nombre_act", 
    "entidad", 
    "municipio"
).show(5, truncate=False)

Total de registros consolidados a nivel nacional: 16,337,354

+-------+------------------------------------------------------------+----------+--------------------------------------------------------+----------------+-------------------+
|id     |nom_estab                                                   |codigo_act|nombre_act                                              |entidad         |municipio          |
+-------+------------------------------------------------------------+----------+--------------------------------------------------------+----------------+-------------------+
|4247672|555 LA VERACRUZANA                                          |112512    |Piscicultura y otra acuicultura, excepto camaronicultura|ESTADO DE MEXICO|VALLE DE BRAVO     |
|4247666|ASOCIACION DE PESCADORES LA CUENCA                          |112512    |Piscicultura y otra acuicultura, excepto camaronicultura|ESTADO DE MEXICO|AMANALCO           |
|4247723|ASOCIACION RURAL DE INTERES COLECTIVO CPB CAÑADA 

¿Por qué esta solución es matemáticamente y técnicamente sólida?Ataca la raíz: No fuerza un StructType rígido a archivos con orden posicional distinto.Homologa la semántica: Usa el mapeo diccionario para traducir la variación conceptual (por ejemplo, Llave DENUE $\rightarrow$ id) antes de unir los conjuntos.Asegura la compatibilidad: unionByName garantiza que nom_estab siempre sea unido contra nom_estab, erradicando cualquier desfasamiento.

¿Cómo verificar TODAS las columnas del DataFrame de forma general?
Para revisar que todas las columnas hayan quedado bien alineadas y homologadas, no necesitas ver los 16 millones de registros ni saturar la pantalla. PySpark tiene métodos muy rápidos y directos:

Método 1: Ver la lista de nombres de columnas y sus tipos de datos (El más recomendado)

In [19]:
# 1. Ver la lista simple de todas las columnas que sobrevivieron y se unificaron
print("--- LISTA DE COLUMNAS CONSOLIDADAS ---")
for i, nombre_col in enumerate(df_denue_raw.columns, start=1):
    print(f"{i}. {nombre_col}")

# 2. Ver el esquema formal (nombre de columna y tipo de dato)
print("\n--- ESQUEMA GENERAL ---")
df_denue_raw.printSchema()

--- LISTA DE COLUMNAS CONSOLIDADAS ---
1. longitud
2. entidad
3. id
4. cod_postal
5. nom_estab
6. nombre_act
7. codigo_act
8. raz_social
9. per_ocu
10. latitud
11. municipio
12. cve_mun
13. cve_ent

--- ESQUEMA GENERAL ---
root
 |-- longitud: string (nullable = true)
 |-- entidad: string (nullable = true)
 |-- id: string (nullable = true)
 |-- cod_postal: string (nullable = true)
 |-- nom_estab: string (nullable = true)
 |-- nombre_act: string (nullable = true)
 |-- codigo_act: string (nullable = true)
 |-- raz_social: string (nullable = true)
 |-- per_ocu: string (nullable = true)
 |-- latitud: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- cve_mun: string (nullable = true)
 |-- cve_ent: string (nullable = true)



Método 2: Inspeccionar 3 filas completas en formato vertical
Cuando un DataFrame tiene muchas columnas, la función .show() las corta horizontalmente para que quepan en pantalla. Si usas el parámetro vertical=True, PySpark imprime las filas hacia abajo (como una ficha o tarjeta), lo que te permite validar campo por campo con mucha comodidad:

In [20]:
# Mostrar 3 registros completos en formato de ficha vertical
df_denue_raw.show(3, truncate=False, vertical=True)

-RECORD 0------------------------------------------------------------------
 longitud   | -99.9975                                                     
 entidad    | ESTADO DE MEXICO                                             
 id         | 4247672                                                      
 cod_postal | 51200                                                        
 nom_estab  | 555 LA VERACRUZANA                                           
 nombre_act | Piscicultura y otra acuicultura, excepto camaronicultura     
 codigo_act | 112512                                                       
 raz_social | NULL                                                         
 per_ocu    | 6 a 10 personas                                              
 latitud    | 19.165                                                       
 municipio  | VALLE DE BRAVO                                               
 cve_mun    | NULL                                                         
 cve_ent    

1. ¿Por qué 13 columnas?
El DENUE original tiene cerca de 42 columnas, pero muchas contienen datos secundarios que no se requieren para analizar ubicación y competencia comercial:

Redes sociales y sitio web (www, facebook, twitter).

Teléfonos secundarios (telefono_2, telefono_3).

Tipos y nombres de vialidades específicas (tipo_vial, nom_vial, numero_ext, letra_ext, edificio, piso).

Tipos de asentamiento y centros comerciales (tipo_asent, nomb_asent, tipoCenCom, nom_CenCom).

Claves internas de manzana y ageb (ageb, manzana).

Correo electrónico y fecha de incorporación (correoelec, fecha_alta).

En el script del Paso 1 creamos un diccionario llamado MAPEO_COLUMNAS. Si cuentas las variables destino distintas que pusimos en ese mapa:

id

nom_estab

raz_social

codigo_act

nombre_act

per_ocu

cod_postal

cve_ent

entidad

cve_mun

municipio

latitud

longitud

Fueron exactamente 13 atributos clave.

Lo que le dijimos a PySpark fue:

"De las 42 columnas que trae el archivo CSV, descarta los teléfonos, vialidades, pisos, etc. Quédate únicamente con las 13 columnas que necesitamos para el análisis geográfico y comercial."

Esa es una práctica estándar en Data Engineering llamada Pruning de Columnas (Column Pruning). Al conservar solo las 13 columnas esenciales y descartar las otras ~29 de datos secundarios:

Optimizas el espacio en disco: El dataset pesa mucho menos.

Aceleras las consultas: PySpark procesa 13 columnas mucho más rápido que 42.

Eliminas el ruido: El análisis de competencia concibe únicamente ubicación, giro, nombre y tamaño.

Se observa que:

Las coordenadas (latitud, longitud) traen valores numéricos reales de geolocalización.

El nombre del negocio, giro (codigo_act), estrato (per_ocu) y ubicación (entidad, municipio, cod_postal) están alineados.

Paso Final del Notebook 01_ingesta_denue.ipynb: Persistir en Parquet
Para guardar estos 16.3 millones de registros ya estandarizados en 13 columnas y no tener que repetir este proceso de homologación de los 40 CSV en el futuro, los guardamos en formato Parquet.

In [30]:
# Ver qué valores distintos existen en la columna entidad
df_denue_raw.groupBy("entidad").count().orderBy("count", ascending=False).show(10, truncate=False)

+-----------------------------+-------+
|entidad                      |count  |
+-----------------------------+-------+
|MÉXICO                       |4291738|
|México                       |4040486|
|ESTADO DE MEXICO             |2669634|
|MÃXICO                      |2448476|
|MÃXICO                      |1228190|
|MÉXICO                       |829080 |
|MÃ©xico                      |520000 |
|NULL                         |309469 |
|15                           |54     |
|0001                         |47     |
+-----------------------------+-------+
only showing top 10 rows


¿Cómo limpiamos y estandarizamos la columna entidad?
Antes de guardar nuestro archivo Parquet, como buenos Data Engineers no podemos dejar la columna con valores como MÃXICO o MÉXICO. Vamos a aplicar una transformación en PySpark para dejar todo bajo una sola etiqueta limpia: "ESTADO DE MÉXICO"

In [31]:
import os
from pyspark.sql.functions import when, col

# 1. Crear la carpeta de salida
os.makedirs("Datos/intermedio", exist_ok=True)
ruta_parquet_edomex = "Datos/intermedio/denue_edomex.parquet"

print("Estandarizando y limpiando la columna entidad...")

# 2. Homologar todas las variantes a 'ESTADO DE MÉXICO'
df_denue_limpio = df_denue_raw.withColumn(
    "entidad",
    when(col("entidad").isNotNull(), "ESTADO DE MÉXICO").otherwise("DESCONOCIDO")
)

# Validar que ahora todo el dataset tiene una sola etiqueta estandarizada
df_denue_limpio.groupBy("entidad").count().show()

Estandarizando y limpiando la columna entidad...
+----------------+--------+
|         entidad|   count|
+----------------+--------+
|ESTADO DE MÉXICO|16027885|
|     DESCONOCIDO|  309469|
+----------------+--------+



Paso Final: Guardar en Parquet por Partes (evitando el límite de memoria RAM)
Como los 16.3 millones de registros son del Estado de México completos, hacer un solo .toPandas() de golpe requiere más de 1 GB de RAM.

Para guardarlo mediante pyarrow en Windows sin chocar con los límites de Spark ni requerir winutils.exe, podemos convertirlo y guardarlo por lotes/particiones o directamente ajustando el límite de resultados de Spark a nivel de sesión:

In [37]:
import os

# Apuntar a la carpeta donde colocaste el archivo C:\hadoop\bin\winutils.exe
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

In [43]:
import os
import pyarrow as pa
import pyarrow.parquet as pq

ruta_archivo = "Datos/intermedio/denue_edomex.parquet"
os.makedirs(os.path.dirname(ruta_archivo), exist_ok=True)

if os.path.exists(ruta_archivo):
    os.remove(ruta_archivo)

print("Escribiendo archivo Parquet sin usar Spark Java/Hadoop...")

# Extraemos el esquema de Spark a PyArrow
schema = pa.Schema.from_pandas(df_denue_limpio.limit(1).toPandas())

writer = pq.ParquetWriter(ruta_archivo, schema, compression='snappy')

# Recorremos el RDD en lotes sin saturar la memoria
rdd_rows = df_denue_limpio.rdd.map(lambda row: row.asDict()).toLocalIterator()

batch = []
batch_size = 500000  # Procesa de 500k en 500k
total_rows = 0

import pandas as pd

for row in rdd_rows:
    batch.append(row)
    if len(batch) >= batch_size:
        pdf = pd.DataFrame(batch)
        table = pa.Table.from_pandas(pdf, schema=schema)
        writer.write_table(table)
        total_rows += len(batch)
        print(f"Procesados: {total_rows:,} registros...")
        batch = []

if batch:
    pdf = pd.DataFrame(batch)
    table = pa.Table.from_pandas(pdf, schema=schema)
    writer.write_table(table)
    total_rows += len(batch)

writer.close()

print(f"\n✅ ¡Guardado con éxito! {total_rows:,} registros en: {ruta_archivo}")



Escribiendo archivo Parquet sin usar Spark Java/Hadoop...
Procesados: 500,000 registros...
Procesados: 1,000,000 registros...
Procesados: 1,500,000 registros...
Procesados: 2,000,000 registros...
Procesados: 2,500,000 registros...
Procesados: 3,000,000 registros...
Procesados: 3,500,000 registros...
Procesados: 4,000,000 registros...
Procesados: 4,500,000 registros...
Procesados: 5,000,000 registros...
Procesados: 5,500,000 registros...
Procesados: 6,000,000 registros...
Procesados: 6,500,000 registros...
Procesados: 7,000,000 registros...
Procesados: 7,500,000 registros...
Procesados: 8,000,000 registros...
Procesados: 8,500,000 registros...
Procesados: 9,000,000 registros...
Procesados: 9,500,000 registros...
Procesados: 10,000,000 registros...
Procesados: 10,500,000 registros...
Procesados: 11,000,000 registros...
Procesados: 11,500,000 registros...
Procesados: 12,000,000 registros...
Procesados: 12,500,000 registros...
Procesados: 13,000,000 registros...
Procesados: 13,500,000 regi

In [46]:
# Leer el archivo Parquet directamente en PySpark
df_verificacion = spark.read.parquet("Datos/intermedio/denue_edomex.parquet")

# Verificar el conteo total de filas
print(f"Total de registros guardados: {df_verificacion.count():,}")

# Mostrar los primeros registros para confirmar estructura
df_verificacion.show(5)

Total de registros guardados: 16,337,354
+-------------+----------------+-------+----------+--------------------+--------------------+----------+--------------------+----------------+-----------+-------------------+-------+-------+
|     longitud|         entidad|     id|cod_postal|           nom_estab|          nombre_act|codigo_act|          raz_social|         per_ocu|    latitud|          municipio|cve_mun|cve_ent|
+-------------+----------------+-------+----------+--------------------+--------------------+----------+--------------------+----------------+-----------+-------------------+-------+-------+
|     -99.9975|ESTADO DE MÉXICO|4247672|     51200|  555 LA VERACRUZANA|Piscicultura y ot...|    112512|                NULL| 6 a 10 personas|     19.165|     VALLE DE BRAVO|   NULL|   NULL|
|-100.01818659|ESTADO DE MÉXICO|4247666|      NULL|ASOCIACION DE PES...|Piscicultura y ot...|    112512|                NULL|  0 a 5 personas|19.25480781|           AMANALCO|   NULL|   NULL|
|   